# CIFAR-10 Image Classification with Transfer Learning

**Objective:** train and evaluate an ImageNet-pretrained ResNet18 using a leakage-safe CIFAR-10 workflow. This notebook delegates reusable logic to `src/` so notebook and command-line behavior stay consistent.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import sys, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torchvision import datasets

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from dataset import CIFAR10_CLASSES, create_dataloaders
from utils import build_model, set_seed
from train import train
from evaluate import evaluate
import argparse
set_seed(42)
SMOKE_TEST = os.getenv('SMOKE_TEST', '0') == '1'
print('Smoke mode:', SMOKE_TEST)
print('PyTorch:', torch.__version__, '| Root:', ROOT)

## 2. Dataset loading and exploration

CIFAR-10 has 60,000 public 32×32 RGB images across ten approximately balanced classes. The scripts automatically download it. We split the official 50,000-image training set into 45,000 train and 5,000 validation images; the official 10,000-image test set is retained.

In [ ]:
raw_train = datasets.CIFAR10(root=ROOT/'data/raw', train=True, download=True)
raw_test = datasets.CIFAR10(root=ROOT/'data/raw', train=False, download=True)
print('Official training images:', len(raw_train))
print('Official test images:', len(raw_test))
print('Classes:', raw_train.classes)
print('Image dimensions/mode:', raw_train[0][0].size, raw_train[0][0].mode)

In [ ]:
counts = pd.Series(raw_train.targets).value_counts().sort_index()
distribution = pd.DataFrame({'class': raw_train.classes, 'images': counts.values})
distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=distribution, x='class', y='images', color='#2783DE', ax=ax)
ax.set(title='CIFAR-10 official training class distribution', xlabel='Class', ylabel='Number of images')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
rng = np.random.default_rng(42)
indices = rng.choice(len(raw_train), 12, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(10, 7))
for ax, idx in zip(axes.flat, indices):
    image, label = raw_train[int(idx)]
    ax.imshow(image); ax.set_title(raw_train.classes[label]); ax.axis('off')
fig.suptitle('Sample CIFAR-10 images'); plt.tight_layout(); plt.show()

## 3. Preprocessing, augmentation, and DataLoaders

Training uses resize, random resized crop, horizontal flip, small rotation, tensor conversion, and ImageNet normalization. Validation/test are deterministic, so augmented views never leak into evaluation.

In [ ]:
bundle = create_dataloaders(ROOT/'data/raw', batch_size=64, image_size=224, seed=42)
print({'train': len(bundle.train_indices), 'validation': len(bundle.val_indices), 'test': len(bundle.test_indices)})
images, labels = next(iter(bundle.train_loader))
print('Batch tensor:', tuple(images.shape), '| Labels:', tuple(labels.shape))

## 4. Transfer learning model

Transfer learning reuses visual features learned on ImageNet. The ResNet18 backbone is initially frozen and its 1,000-output layer is replaced by a ten-output classifier. `train.py` optionally unfreezes the final residual block for low-rate fine-tuning.

In [ ]:
model = build_model(num_classes=10, pretrained=True, freeze_backbone=True)
print(model.fc)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('Total parameters:', sum(p.numel() for p in model.parameters()))

## 5. Training and validation

The full configuration below trains for ten epochs, checkpoints the minimum validation loss, applies learning-rate reduction and early stopping, and saves measured curves. Adjust epochs only when intentionally running a smoke test.

In [ ]:
train_args = argparse.Namespace(
    epochs=1 if SMOKE_TEST else 10, batch_size=16 if SMOKE_TEST else 64, lr=1e-3, seed=42, image_size=64 if SMOKE_TEST else 224, val_fraction=.1,
    model_path=str(ROOT/'models/best_model.pth'), data_dir=str(ROOT/'data/raw'),
    results_dir=str(ROOT/'results'), num_workers=0, device='auto', patience=3,
    fine_tune_epoch=0 if SMOKE_TEST else 4, no_pretrained=False, max_train_samples=128 if SMOKE_TEST else None,
    max_val_samples=64 if SMOKE_TEST else None, max_test_samples=64 if SMOKE_TEST else None
)
history = train(train_args)

## 6. Training curves

In [ ]:
history_df = pd.DataFrame(history, index=range(1, len(history['train_loss'])+1))
display(history_df)
from IPython.display import Image as IPImage, display
display(IPImage(filename=str(ROOT/'results/training_curves.png')))

Interpret the observed curves rather than assuming success: a growing train/validation gap indicates overfitting; uniformly poor curves indicate underfitting; large oscillations indicate instability.

## 7. Test evaluation, confusion matrix, and classification report

In [ ]:
eval_args = argparse.Namespace(model_path=str(ROOT/'models/best_model.pth'), data_dir=str(ROOT/'data/raw'),
    results_dir=str(ROOT/'results'), batch_size=16 if SMOKE_TEST else 64, seed=42, num_workers=0, device='auto', max_test_samples=64 if SMOKE_TEST else None)
metrics = evaluate(eval_args)
pd.Series(metrics)

In [ ]:
display(IPImage(filename=str(ROOT/'results/confusion_matrix.png')))
print((ROOT/'results/classification_report.txt').read_text())

Accuracy is overall correctness. Precision measures the correctness of positive predictions, recall measures recovered true examples, and F1 balances both. Macro averages weight classes equally; weighted averages account for support.

## 8. Sample predictions and error analysis

In [ ]:
predictions = pd.read_csv(ROOT/'results/predictions.csv')
correct_ids = predictions.index[predictions.actual == predictions.predicted].tolist()[:4]
wrong_ids = predictions.index[predictions.actual != predictions.predicted].tolist()[:4]
selected = correct_ids + wrong_ids
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for ax, idx in zip(axes.flat, selected):
    image, _ = raw_test[int(idx)]
    row = predictions.iloc[int(idx)]
    ax.imshow(image); ax.axis('off')
    ax.set_title(f"Actual: {row.actual}\nPred: {row.predicted} ({row.confidence:.1%})",
                 color='green' if row.actual == row.predicted else 'firebrick')
for ax in axes.flat[len(selected):]: ax.axis('off')
fig.suptitle('Measured correct and incorrect test predictions'); plt.tight_layout(); plt.show()
if not wrong_ids: print('No incorrect examples occurred in the inspected predictions; none were fabricated.')

Misclassifications should be inspected individually. Plausible causes include 32×32 source resolution, visual similarity among animal classes, clutter, pose, and transfer-domain mismatch; these are hypotheses until confirmed by the displayed examples.

## 9. Responsible AI

- **Bias:** balanced labels do not eliminate selection or representation bias.
- **Fairness:** compare per-class recall/F1; aggregate accuracy can hide weak classes.
- **Privacy:** CIFAR-10 is a public benchmark. Human-image deployments require consent, minimization, retention controls, and review.
- **Reliability:** softmax values are not guarantees; blur and distribution shift can fail.
- **Appropriate use:** education and low-risk prototypes only—not surveillance, identity, safety-critical, or high-stakes decisions.

## 10. Conclusion

This notebook executes the same tested pipeline as the command-line tools. Final conclusions must reference the metrics and figures generated by the current run; smoke-run results must not be presented as final performance.